<a href="https://colab.research.google.com/github/AkhyanSandyAfgara/AI-Praktikum1/blob/main/ANN_heart.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Klasifikasi Penyakit Jantung Menggunakan Jaringan Saraf Tiruan

Dataset yang digunakan adalah **Heart Disease Dataset** dengan 1025 data pasien dan 13 fitur klinis. Tujuannya adalah memprediksi apakah seorang pasien menderita penyakit jantung (target = 1) atau tidak (target = 0).

---

## Langkah 1: Mengimpor Pustaka yang Diperlukan

Kita akan menggunakan NumPy untuk menangani komputasi numerik secara efisien, serta Pandas untuk membaca dan memproses dataset.

In [1]:
import numpy as np
import pandas as pd

## Langkah 2: Menginisialisasi Jaringan Saraf Tiruan

* Menetapkan bobot dan bias awal untuk jaringan saraf dua lapis.
* Menggunakan `np.random.seed(42)` untuk hasil yang dapat direproduksi.
* Bobot (W1, W2) diinisialisasi dengan nilai acak kecil yang diskalakan dengan 0.01 untuk menghindari bobot awal yang besar.
* Bentuk W1: (ukuran lapisan tersembunyi, ukuran lapisan masukan).
* Bentuk W2: (ukuran lapisan keluaran, ukuran lapisan tersembunyi).
* Bias (b1, b2) diinisialisasi ke vektor nol yang sesuai dengan ukuran lapisannya.

In [2]:
def initialize_parameters(input_size, hidden_size, output_size):
    np.random.seed(42)
    parameters = {
        "W1": np.random.randn(hidden_size, input_size) * 0.01,
        "b1": np.zeros((hidden_size, 1)),
        "W2": np.random.randn(output_size, hidden_size) * 0.01,
        "b2": np.zeros((output_size, 1))
    }
    return parameters

## Langkah 3: Mendefinisikan Fungsi Aktivasi

Fungsi aktivasi memperkenalkan non-linearitas ke dalam model, membantunya mempelajari pola-pola kompleks. Di sini kita menggunakan:
* **ReLU** untuk lapisan tersembunyi — cocok menangani fitur klinis yang beragam.
* **Sigmoid** untuk lapisan keluaran — menghasilkan probabilitas antara 0 dan 1 untuk klasifikasi biner.

In [3]:
def sigmoid(Z):
    return 1 / (1 + np.exp(-Z))


def relu(Z):
    return np.maximum(0, Z)


def relu_derivative(Z):
    return (Z > 0).astype(int)

## Langkah 4: Perambatan Maju (Forward Propagation)

Dalam propagasi maju, fungsi menghitung keluaran jaringan saraf untuk masukan X dan parameter tertentu.
* Pertama, algoritma menghitung kombinasi linier Z1 untuk lapisan tersembunyi dengan mengalikan input X dengan bobot W1 dan menambahkan bias b1.
* Kemudian, fungsi aktivasi ReLU diterapkan pada Z1 untuk menghasilkan aktivasi lapisan tersembunyi A1.
* Selanjutnya, algoritma menghitung kombinasi linier Z2 untuk lapisan keluaran dengan mengalikan A1 dengan W2 dan menambahkan b2.
* Fungsi aktivasi sigmoid diterapkan pada Z2 untuk menghasilkan keluaran akhir A2 (probabilitas penyakit jantung).
* Fungsi mengembalikan output A2 beserta cache yang berisi nilai-nilai sementara yang dibutuhkan untuk backpropagation.

In [4]:
def forward_propagation(X, parameters):
    W1, b1, W2, b2 = parameters["W1"], parameters["b1"], parameters["W2"], parameters["b2"]

    Z1 = np.dot(W1, X) + b1
    A1 = relu(Z1)
    Z2 = np.dot(W2, A1) + b2
    A2 = sigmoid(Z2)

    cache = {"Z1": Z1, "A1": A1, "Z2": Z2, "A2": A2}
    return A2, cache

## Langkah 5: Menghitung Biaya (Cost Function)

Fungsi biaya menghitung kerugian entropi silang biner (*binary cross-entropy*) yang mengukur seberapa baik prediksi jaringan saraf A2 sesuai dengan label sebenarnya Y.

* m adalah jumlah contoh (jumlah pasien).
* `np.squeeze` menghilangkan dimensi tambahan apa pun, mengembalikan biaya sebagai nilai skalar.
* Semakin kecil nilai biaya, semakin baik model memprediksi penyakit jantung.

In [5]:
def compute_cost(Y, A2):
    m = Y.shape[1]
    cost = -np.sum(Y * np.log(A2 + 1e-8) + (1 - Y) * np.log(1 - A2 + 1e-8)) / m
    return np.squeeze(cost)

## Langkah 6: Backpropagation

Backpropagation menghitung gradien yang dibutuhkan untuk memperbarui parameter jaringan selama pelatihan.

* Algoritma menghitung kesalahan pada lapisan keluaran (dZ2) sebagai selisih antara keluaran yang diprediksi (A2) dan label sebenarnya (Y).
* Dengan menggunakan kesalahan ini, algoritma menghitung gradien bobot (dW2) dan bias (db2) untuk lapisan keluaran.
* Kemudian, algoritma menyebarkan kesalahan kembali ke lapisan tersembunyi dengan mengalikannya dengan transpose dari W2 dan secara elemen demi elemen dengan turunan dari aktivasi ReLU.
* Terakhir, ia menghitung gradien untuk bobot lapisan tersembunyi (dW1) dan bias (db1).
* Semua gradien dirata-ratakan berdasarkan jumlah contoh m untuk memastikan pembaruan yang stabil.

In [6]:
def backward_propagation(X, Y, parameters, cache):
    m = X.shape[1]
    W2 = parameters["W2"]

    dZ2 = cache["A2"] - Y
    dW2 = np.dot(dZ2, cache["A1"].T) / m
    db2 = np.sum(dZ2, axis=1, keepdims=True) / m

    dZ1 = np.dot(W2.T, dZ2) * relu_derivative(cache["Z1"])
    dW1 = np.dot(dZ1, X.T) / m
    db1 = np.sum(dZ1, axis=1, keepdims=True) / m

    grads = {"dW1": dW1, "db1": db1, "dW2": dW2, "db2": db2}
    return grads

## Langkah 7: Memperbarui Parameter

Gradient descent memperbarui parameter menggunakan gradien yang dihitung dan laju pembelajaran (*learning rate*). Semakin besar learning rate, semakin cepat model belajar, namun berisiko tidak konvergen.

In [7]:
def update_parameters(parameters, grads, learning_rate):
    for key in parameters.keys():
        parameters[key] -= learning_rate * grads["d" + key]
    return parameters

## Langkah 8: Melatih Jaringan Saraf Tiruan

Kami melatih jaringan saraf melalui beberapa iterasi (epoch), memperbarui parameter menggunakan backpropagation dan gradient descent. Untuk dataset Heart Disease:

* **input_size = 13** (jumlah fitur klinis: usia, jenis kelamin, tekanan darah, kolesterol, dll.)
* **hidden_size = 16** (lapisan tersembunyi dengan 16 neuron)
* **output_size = 1** (prediksi biner: penyakit jantung atau tidak)

In [8]:
def train_neural_network(X, Y, input_size, hidden_size, output_size, epochs=1000, learning_rate=0.01):
    parameters = initialize_parameters(input_size, hidden_size, output_size)

    for i in range(epochs):
        A2, cache = forward_propagation(X, parameters)
        cost = compute_cost(Y, A2)
        grads = backward_propagation(X, Y, parameters, cache)
        parameters = update_parameters(parameters, grads, learning_rate)

        if i % 100 == 0:
            print(f"Epoch {i}: Cost = {cost:.6f}")

    return parameters

## Langkah 9: Membuat Prediksi

Model yang telah dilatih memprediksi output dengan melakukan propagasi maju dan menerapkan ambang batas 0.5.
* Jika A2 > 0.5 → prediksi **1** (pasien menderita penyakit jantung)
* Jika A2 ≤ 0.5 → prediksi **0** (pasien tidak menderita penyakit jantung)

In [9]:
def predict(X, parameters):
    A2, _ = forward_propagation(X, parameters)
    return (A2 > 0.5).astype(int)

## Langkah 10: Menguji Model dengan Dataset Heart Disease

Kami menguji model menggunakan dataset penyakit jantung. Dataset dibagi menjadi **data latih (80%)** dan **data uji (20%)**. Sebelum pelatihan, fitur dinormalisasi menggunakan standardisasi (z-score) agar setiap fitur memiliki skala yang seragam — penting karena fitur seperti kolesterol (200-an) jauh lebih besar dari fitur biner (0/1).

In [10]:
# ── 1. Load dataset ──────────────────────────────────────────────────────────
df = pd.read_csv("heart.csv")
print("Bentuk dataset:", df.shape)
print("\nLima baris pertama:")
print(df.head())
print("\nDistribusi target (0 = Sehat, 1 = Penyakit Jantung):")
print(df["target"].value_counts())

Bentuk dataset: (1025, 14)

Lima baris pertama:
   age  sex  cp  trestbps  chol  fbs  restecg  thalach  exang  oldpeak  slope  \
0   52    1   0       125   212    0        1      168      0      1.0      2   
1   53    1   0       140   203    1        0      155      1      3.1      0   
2   70    1   0       145   174    0        1      125      1      2.6      0   
3   61    1   0       148   203    0        1      161      0      0.0      2   
4   62    0   0       138   294    1        1      106      0      1.9      1   

   ca  thal  target  
0   2     3       0  
1   0     3       0  
2   0     3       0  
3   1     3       0  
4   3     2       0  

Distribusi target (0 = Sehat, 1 = Penyakit Jantung):
target
1    526
0    499
Name: count, dtype: int64


In [11]:
# ── 2. Preprocessing ─────────────────────────────────────────────────────────
X_raw = df.drop("target", axis=1).values  # shape: (1025, 13)
Y_raw = df["target"].values               # shape: (1025,)

X_mean = X_raw.mean(axis=0)
X_std  = X_raw.std(axis=0)
X_norm = (X_raw - X_mean) / (X_std + 1e-8)

np.random.seed(42)
m_total = X_norm.shape[0]
indices = np.random.permutation(m_total)
split   = int(0.8 * m_total)

train_idx, test_idx = indices[:split], indices[split:]

# Transpose agar bentuk sesuai jaringan: (fitur, sampel)
X_train = X_norm[train_idx].T   # (13, 820)
Y_train = Y_raw[train_idx].reshape(1, -1)   # (1, 820)
X_test  = X_norm[test_idx].T    # (13, 205)
Y_test  = Y_raw[test_idx].reshape(1, -1)    # (1, 205)

print(f"Data latih  : X={X_train.shape}, Y={Y_train.shape}")
print(f"Data uji    : X={X_test.shape},  Y={Y_test.shape}")

Data latih  : X=(13, 820), Y=(1, 820)
Data uji    : X=(13, 205),  Y=(1, 205)


In [12]:
# ── 3. Latih model ───────────────────────────────────────────────────────────
print("=" * 55)
print(" Melatih Jaringan Saraf Tiruan — Heart Disease Dataset")
print("=" * 55)

trained_parameters = train_neural_network(
    X_train, Y_train,
    input_size=13,
    hidden_size=16,
    output_size=1,
    epochs=1000,
    learning_rate=0.05
)

 Melatih Jaringan Saraf Tiruan — Heart Disease Dataset
Epoch 0: Cost = 0.693134
Epoch 100: Cost = 0.684929
Epoch 200: Cost = 0.560573
Epoch 300: Cost = 0.365977
Epoch 400: Cost = 0.330796
Epoch 500: Cost = 0.321768
Epoch 600: Cost = 0.317218
Epoch 700: Cost = 0.313690
Epoch 800: Cost = 0.311032
Epoch 900: Cost = 0.308139


In [13]:
# ── 4. Evaluasi model ────────────────────────────────────────────────────────
def accuracy(Y_true, Y_pred):
    return np.mean(Y_true == Y_pred) * 100

pred_train = predict(X_train, trained_parameters)
pred_test  = predict(X_test,  trained_parameters)

acc_train = accuracy(Y_train, pred_train)
acc_test  = accuracy(Y_test,  pred_test)

print("\n" + "=" * 40)
print("         HASIL EVALUASI MODEL")
print("=" * 40)
print(f"Akurasi Data Latih : {acc_train:.2f}%")
print(f"Akurasi Data Uji   : {acc_test:.2f}%")
print("=" * 40)

print("\nContoh 10 prediksi pertama pada data uji:")
print(f"  Label asli   : {Y_test[0, :10].tolist()}")
print(f"  Prediksi     : {pred_test[0, :10].tolist()}")


         HASIL EVALUASI MODEL
Akurasi Data Latih : 86.95%
Akurasi Data Uji   : 84.39%

Contoh 10 prediksi pertama pada data uji:
  Label asli   : [0, 1, 0, 1, 0, 1, 1, 1, 0, 1]
  Prediksi     : [0, 1, 0, 0, 0, 1, 1, 1, 0, 1]


## Analisis

Berfokus pada pengembangan model klasifikasi penyakit jantung menggunakan arsitektur Jaringan Saraf Tiruan dua lapis yang dibangun secara manual berbasis NumPy, tanpa mengandalkan library high-level. Dengan memanfaatkan dataset heart.csv yang mencakup 1.025 data pasien dan 13 indikator medis, model ini dirancang untuk memprediksi risiko penyakit jantung melalui fungsi aktivasi ReLU dan Sigmoid. Tahapan krusial seperti normalisasi fitur melalui teknik z-score juga telah diterapkan guna menyamakan skala data sebelum proses pelatihan dilakukan selama 1.000 epoch. Berdasarkan hasil evaluasi pada data uji, model menunjukkan performa yang stabil dengan penurunan cost function yang konsisten, yang membuktikan keberhasilan penerapan teori jaringan saraf tiruan dalam menangkap pola klinis yang kompleks untuk prediksi kesehatan.

## Kesimpulan

Jaringan saraf tiruan dua lapis berhasil diterapkan pada **Heart Disease Dataset** dengan 1025 data pasien dan 13 fitur klinis.

* **Preprocessing** penting dilakukan — normalisasi z-score memastikan fitur dengan skala berbeda (kolesterol, tekanan darah, usia) tidak mendominasi proses pembelajaran.
* Model dimulai dengan bobot acak kecil. Selama 1000 epoch, gradient descent secara bertahap mengoptimalkan W1, b1, W2, dan b2 sehingga fungsi biaya terus menurun.
* Fungsi aktivasi **ReLU** pada lapisan tersembunyi membantu model menangkap pola non-linier dari data klinis, sementara **Sigmoid** pada lapisan keluaran menghasilkan probabilitas yang dapat diinterpretasikan sebagai risiko penyakit jantung.
* Hasil prediksi pada data uji membuktikan bahwa jaringan saraf mampu menggeneralisasi pola dari data latih untuk mengklasifikasikan pasien baru yang belum pernah dilihat sebelumnya.